# Week 4 lab

Part 1: Data Preparation

In [ ]:
import numpy as np
import odc.stac
import pandas as pd
import planetary_computer
import pystac_client
import xarray as xr
from dask.distributed import Client
from pystac.extensions.eo import EOExtension as eo
from dask_ml.cluster import SpectralClustering
import pyproj
import panel as pn

# Viz
import hvplot.xarray

In [33]:
pn.extension()

In [3]:
#connecting to planetary computer STAC API
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

#boundary box parameters
bbox = [-118.89, 38.54, -118.57, 38.84]  # Region over a lake in Nevada, USA
datetime = "2017-06-01/2017-09-30"  # Summer months of 2017
collection = "landsat-c2-l2"
platform = "landsat-8"
cloudy_less_than = 1  # percent

#only taking the ones that have low cloud cover for more accurate
search = catalog.search(
    collections=["landsat-c2-l2"],
    bbox=bbox,
    datetime=datetime,
    query={"eo:cloud_cover": {"lt": cloudy_less_than}, "platform": {"in": [platform]}},
)
items = search.get_all_items()
print(f"Returned {len(items)} Items:")
[[i, item.id] for i, item in enumerate(items)]

f:\GitHub\musa-650-spring2026\.venv\Lib\site-packages\pystac_client\item_search.py:896: FutureWarning: get_all_items() is deprecated, use item_collection() instead.
  warnings.warn(


Returned 3 Items:


[[0, 'LC08_L2SP_042033_20170718_02_T1'],
 [1, 'LC08_L2SP_042033_20170702_02_T1'],
 [2, 'LC08_L2SP_042033_20170616_02_T1']]

In [17]:
item = items[1]

In [18]:
assets = []
for _, asset in item.assets.items():
    try:
        assets.append(asset.extra_fields["eo:bands"][0])
    except:
        pass

cols_ordered = [
    "common_name",
    "description",
    "name",
    "center_wavelength",
    "full_width_half_max",
]
bands = pd.DataFrame.from_dict(assets)[cols_ordered]
bands

,common_name,description,name,center_wavelength,full_width_half_max
0,red,Visible red,OLI_B4,0.65,0.04
1,blue,Visible blue,OLI_B2,0.48,0.06
2,green,Visible green,OLI_B3,0.56,0.06
3,nir08,Near infrared,OLI_B5,0.87,0.03
4,lwir11,Long-wave infrared,TIRS_B10,10.90,0.59
5,swir16,Short-wave infrared,OLI_B6,1.61,0.09
6,swir22,Short-wave infrared,OLI_B7,2.20,0.19
7,coastal,Coastal/Aerosol,OLI_B1,0.44,0.02


In [19]:
ds_2017 = odc.stac.stac_load(
    [item],
    bands=bands.common_name.values,
    bbox=bbox,
    chunks={},  # <-- use Dask
).isel(time=0)

retain crs Attribute

In [45]:
prop = item.properties
print(prop)

{'gsd': 30, 'created': '2022-05-06T17:46:34.110946Z', 'sci:doi': '10.5066/P9OGBGM6', 'datetime': '2017-07-02T18:33:06.200763Z', 'platform': 'landsat-8', 'proj:shape': [7941, 7811], 'description': 'Landsat Collection 2 Level-2', 'instruments': ['oli', 'tirs'], 'eo:cloud_cover': 0.53, 'proj:transform': [30.0, 0.0, 246285.0, 0.0, -30.0, 4425915.0], 'view:off_nadir': 0, 'landsat:wrs_row': '033', 'landsat:scene_id': 'LC80420332017183LGN00', 'landsat:wrs_path': '042', 'landsat:wrs_type': '2', 'view:sun_azimuth': 125.03739105, 'landsat:correction': 'L2SP', 'view:sun_elevation': 65.85380157, 'landsat:cloud_cover_land': 0.53, 'landsat:collection_number': '02', 'landsat:collection_category': 'T1', 'proj:code': 'EPSG:32611'}


# fix the epsg line below

In [ ]:
epsg = item.properties.get("EPSG:")
ds_2017.attrs["crs"] = f"epsg:{epsg}"

In [49]:
da_2017 = ds_2017.to_array(dim="band")
da_2017

<xarray.DataArray (band: 8, y: 1128, x: 950)> Size: 17MB
dask.array<stack, shape=(8, 1128, 950), dtype=uint16, chunksize=(1, 1128, 950), chunktype=numpy.ndarray>
Coordinates:
  * band         (band) object 64B 'red' 'blue' 'green' ... 'swir22' 'coastal'
  * y            (y) float64 9kB 4.301e+06 4.301e+06 ... 4.267e+06 4.267e+06
  * x            (x) float64 8kB 3.353e+05 3.353e+05 ... 3.637e+05 3.638e+05
    spatial_ref  int32 4B 32611
    time         datetime64[ns] 8B 2017-07-02T18:33:06.200763
Attributes:
    crs:      epsg:None

flatten array to two demensional

In [35]:
flattened_xda = da_2017.stack(z=("x", "y"))  # flatten each band
flattened_t_xda = flattened_xda.transpose("z", "band")
flattened_t_xda

<xarray.DataArray (z: 1071600, band: 8)> Size: 17MB
dask.array<transpose, shape=(1071600, 8), dtype=uint16, chunksize=(1071600, 1), chunktype=numpy.ndarray>
Coordinates:
  * z            (z) object 9MB MultiIndex
  * x            (z) float64 9MB 3.353e+05 3.353e+05 ... 3.638e+05 3.638e+05
  * y            (z) float64 9MB 4.301e+06 4.301e+06 ... 4.267e+06 4.267e+06
  * band         (band) object 64B 'red' 'blue' 'green' ... 'swir22' 'coastal'
    spatial_ref  int32 4B 32611
    time         datetime64[ns] 8B 2017-07-02T18:33:06.200763
Attributes:
    crs:      epsg:None

Standardization

In [36]:
with xr.set_options(keep_attrs=True):
    rescaled_xda = (flattened_t_xda - flattened_t_xda.mean()) / flattened_t_xda.std()
rescaled_xda

<xarray.DataArray (z: 1071600, band: 8)> Size: 69MB
dask.array<truediv, shape=(1071600, 8), dtype=float64, chunksize=(1071600, 1), chunktype=numpy.ndarray>
Coordinates:
  * z            (z) object 9MB MultiIndex
  * x            (z) float64 9MB 3.353e+05 3.353e+05 ... 3.638e+05 3.638e+05
  * y            (z) float64 9MB 4.301e+06 4.301e+06 ... 4.267e+06 4.267e+06
  * band         (band) object 64B 'red' 'blue' 'green' ... 'swir22' 'coastal'
    spatial_ref  int32 4B 32611
    time         datetime64[ns] 8B 2017-07-02T18:33:06.200763
Attributes:
    crs:      epsg:None

## Part 2: K means clustering

In [ ]:
client = Client(processes=False)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://10.102.235.197:8787/status,
Dashboard: http://10.102.235.197:8787/status,Workers: 1
Total threads: 16,Total memory: 31.68 GiB
Status: running,Using processes: False
Comm: inproc://10.102.235.197/29488/1,Workers: 0
Dashboard: http://10.102.235.197:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: inproc://10.102.235.197/29488/4,Total threads: 16
Dashboard: http://10.102.235.197:64374/status,Memory: 31.68 GiB
Nanny: None,


2026-02-05 12:33:13,841 - distributed.scheduler - WARNING - Worker failed to heartbeat for 3028s; removing: <WorkerState 'inproc://10.102.235.197/29488/4', name: 0, status: running, memory: 8, processing: 0>
2026-02-05 12:33:13,848 - distributed.scheduler - WARNING - Workers ['inproc://10.102.235.197/29488/4'] do not use a nanny and will be terminated without restarting them
2026-02-05 12:33:13,852 - distributed.scheduler - WARNING - Removing worker 'inproc://10.102.235.197/29488/4' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('truediv-bac20369fece6711e7a89aa63eb39123', 0, 5), ('truediv-bac20369fece6711e7a89aa63eb39123', 0, 2), ('truediv-bac20369fece6711e7a89aa63eb39123', 0, 1), ('truediv-bac20369fece6711e7a89aa63eb39123', 0, 4), ('truediv-bac20369fece6711e7a89aa63eb39123', 0, 0), ('truediv-bac20369fece6711e7a89aa63eb39123', 0, 7), ('truediv-bac20369fece6711e7a89aa63eb39123', 0, 3), ('truediv-bac20369fece6711e7a89aa63eb39123', 0, 6)} (stimu

In [38]:
X = client.persist(rescaled_xda)
X.shape

(1071600, 8)

K means = 4

In [39]:
clf = SpectralClustering(
    n_clusters=4,
    random_state=0,
    gamma=None,
    kmeans_params={"init_max_iter": 5},
    persist_embedding=True,
)

In [40]:
%time clf.fit(X)

f:\GitHub\musa-650-spring2026\.venv\Lib\site-packages\distributed\client.py:3387: UserWarning: Sending large graph of size 77.83 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


CPU times: total: 3min 21s
Wall time: 1min 25s


,n_clusters,4
,eigen_solver,None
,random_state,0
,n_init,'auto'
,gamma,None
,affinity,'rbf'
,n_neighbors,10
,eigen_tol,0.0
,assign_labels,'kmeans'
,degree,3
,coef0,1


In [41]:
labels = clf.assign_labels_.labels_.compute()
labels.shape

FutureCancelledError: finalize-hlgfinalizecompute-f643f2fc390946568269cd7c48be9811 cancelled for reason: lost dependencies.